# 12 · Benchmark of the MEGAN-SC Python port against the official MEGANv3.21 Fortran

Runs the official MEGANv3.21 binaries (Guenther 2024, Zenodo 10.5281/zenodo.10526206, as shipped in
`MEGANv3.21.tar.gz`) and the MEGAN-SC Python engine on the same inputs and compares them stage by stage.
Needs **Linux x86-64** (Google Colab works; the binaries do not run on Windows outside WSL). About 15-25 min.

| Case | Inputs | What is compared |
|---|---|---|
| S0 harness | official SE_US test case, 2018-05-02 (163 × 163, 12 km Lambert) | our Fortran run vs the reference output shipped in the tarball (proves the binaries and settings are right) |
| SE_US | same | S1 DAYMET, S2 MEGCAN canopy, S3 MEGVEA with the Fortran canopy as input, S4 Python end to end; GAMSM/GAMBD off and on; soil NO (MEGSEA, Yienger-Levy, with and without soil data); speciation and unit chain (mgn2mech, 201 species in t/h) |
| SC | Santa Catarina grid (0.025°, 144 × 224) on 15 Jan and 15 Jul 2019, 2019 monthly LAI (Earth Engine default resampling) and growth forms, idealised diurnal T and PAR | the same stages in the Southern Hemisphere and the production canopy setting, plus day totals weighted with the production EF grid |

Each comparison is made with the engine as used in production, with the two engine patches of `fixes.py` (FIX-6
GAMTP operator precedence, FIX-7 soil-NO air-temperature cap), which has to reproduce the Fortran to float32
precision, and with the engine modules alone, a control showing that the comparison detects a departure from
megvea.f and soilnox.F (rows labelled "original port").

Documented deviations that are kept on purpose (not tested as equal): the Southern-Hemisphere growing season for soil
NO (MEGANv3.21 compares day-of-year with month-day constants, so in SH winter its fertiliser factor goes negative in
crop cells; the port clips it at 0 and production uses the Nov-May season), constant LDF for the four MT classes, and
GAMSM/GAMBD switched on in production.

**Send back:** the folder `RESULTS_DIR` (CSV files + `benchmark_summary.csv`).

In [ ]:
# 1 · Paths ------------------------------------------------------------------------------------------------
# --- Find the MEGAN_SC folder and, in Google Colab, connect Google Drive (the same block opens every notebook) ---------
import os, sys, glob
try:
    from google.colab import drive as _drive
    IN_COLAB = 'MEGAN_SC_RUNNER' not in os.environ          # tools/run_notebook.py sets it when it runs a notebook outside Colab
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    # A failed mount must stop the notebook: otherwise it would read and write a local /content/drive folder that is not
    # Google Drive.
    if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive') and os.listdir('/content/drive'):
        raise RuntimeError('/content/drive holds local files written while Google Drive was not mounted, so Drive cannot be '
                           'mounted there. Use Runtime > Disconnect and delete runtime, then run the notebook again.')
    _drive.mount('/content/drive')


def find_repository():
    """The MEGAN_SC folder: $MEGAN_SC_REPO when set; in Colab, MyDrive/MEGAN_SC (or a MEGAN_SC folder one level down in
    MyDrive, or /content/MEGAN_SC for a clone outside Drive); elsewhere, the current folder or its parent."""
    cands = [os.environ.get('MEGAN_SC_REPO', '')]
    if IN_COLAB:
        cands += ['/content/drive/MyDrive/MEGAN_SC', '/content/drive/MyDrive/megan_sc', '/content/drive/MyDrive/megan-sc']
        cands += sorted(glob.glob('/content/drive/MyDrive/*/MEGAN_SC')) + ['/content/MEGAN_SC']
    cands += [os.getcwd(), os.path.dirname(os.getcwd())]
    for c in cands:
        if c and os.path.isfile(os.path.join(c, 'src', 'megan_sc', 'fixes.py')):
            return os.path.abspath(c)
    raise SystemExit('The MEGAN_SC folder was not found. In Colab, copy the repository to Google Drive as MyDrive/MEGAN_SC '
                     '(tutorial, chapter 3), or run  os.environ["MEGAN_SC_REPO"] = "/path/to/MEGAN_SC"  before this cell.')


REPO_DIR = find_repository()
if os.path.join(REPO_DIR, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('MEGAN_SC folder:', REPO_DIR, '| Colab:', IN_COLAB)

import os, sys, glob, subprocess, shutil, hashlib, json
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    # A failed mount must stop the notebook: if it is hidden, the notebook reads and writes a local /content/drive
    # folder that is not Google Drive (notebook 13 then stops with "found 0").
    if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive') and os.listdir('/content/drive'):
        raise RuntimeError('/content/drive holds local files written while Google Drive was not mounted, so Drive cannot be '
                           'mounted there. Use Runtime > Disconnect and delete runtime, then run the notebook again from cell 1.')
    drive.mount('/content/drive')
MGN_TGZ   = '/content/drive/MyDrive/MEGANv3.21.tar.gz'        # <-- MEGANv3.21.tar.gz from Zenodo 10.5281/zenodo.10526206
MEGAN_DIR = REPO_DIR                      # the repository folder (found by the setup block)
RAW       = '/content/drive/MyDrive/megan_sc_outputs_v10'     # <-- run folder whose 2019 inputs build the SC test case (the
                                                             #     published benchmark used those of an earlier run of notebook 10)
WORK      = '/content/bench'                                  # local scratch (needs ~1.5 GB)
RESULTS_DIR = '/content/drive/MyDrive/megan_sc_benchmark_results'
MGN = os.path.join(WORK, 'MEGANv3.21'); PY = os.path.join(WORK, 'py'); ENGINE_SRC = os.path.join(MEGAN_DIR, 'src')
for d in (WORK, PY, RESULTS_DIR): os.makedirs(d, exist_ok=True)
def sh(cmd, env=None, cwd=None, check=True):
    e = dict(os.environ); e.update(env or {})
    r = subprocess.run(cmd, shell=True, env=e, cwd=cwd or WORK, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if check and r.returncode != 0: raise RuntimeError(f'failed ({r.returncode}): {cmd}')
    return r

In [ ]:
# 2 · Dependencies and the Intel OpenMP runtime the binaries are linked against --------------------------
%pip -q install intel-openmp netCDF4 rasterio geopandas pyyaml pandas
import site, sysconfig
cands = []
for base in {sys.prefix, sys.base_prefix, site.getuserbase(), '/usr/local', '/usr'}:
    cands += glob.glob(os.path.join(base, 'lib', 'libiomp5.so')) + glob.glob(os.path.join(base, 'lib*', '**', 'libiomp5.so'), recursive=True)
assert cands, 'libiomp5.so not found after pip install intel-openmp'
OMP_LIB = os.path.dirname(cands[0]); os.environ['OMP_LIB'] = OMP_LIB
print('libiomp5.so in', OMP_LIB)

In [ ]:
# 3 · Extract the official MEGANv3.21 package (binaries, SE_US inputs, reference output) -----------------
if not os.path.exists(os.path.join(MGN, 'src', 'MEGVEA', 'megvea')):
    sh(f"tar -xzf '{MGN_TGZ}' -C '{WORK}' --exclude='MEGANv3.21/CAMx_ready' --exclude='MEGANv3.21/Output/MEGANv32*'")
for p in ('DAYMET/daymet', 'MEGCAN/megcan', 'MEGSEA/megsea', 'MEGVEA/megvea', 'MGN2MECH/mgn2mech'):
    os.chmod(os.path.join(MGN, 'src', p), 0o755)
sh(f"LD_LIBRARY_PATH={OMP_LIB} ldd {MGN}/src/MEGVEA/megvea | grep -i 'not found' || echo 'all shared libraries resolved'", check=False)
print('tarball sha256[:16]:', hashlib.sha256(open(MGN_TGZ, 'rb').read()).hexdigest()[:16])

In [ ]:
# 4 · Benchmark scripts (benchmark/ of the repository) and the engine patches (identical to notebook 10) ----------
FILES = {
 'ioapi.py': r'''"""Minimal IOAPI-netCDF helpers for the MEGANv3.21 benchmark (read; write a lat-lon IOAPI file)."""
import numpy as np
from netCDF4 import Dataset

def rd(path, var, t=None, squeeze_lay=True):
    with Dataset(path) as d:
        a = np.asarray(d.variables[var][:], dtype=np.float64)          # (TSTEP, LAY, ROW, COL)
    if t is not None: a = a[t]
    if squeeze_lay and a.ndim >= 3 and a.shape[-3] == 1: a = np.squeeze(a, axis=-3)
    return a

def attrs(path):
    with Dataset(path) as d:
        return {k: d.getncattr(k) for k in d.ncattrs()}

def compare(py, ft, mask=None, tiny=1e-12):
    """Agreement statistics between a Python array and the Fortran reference (same shape)."""
    py = np.asarray(py, np.float64); ft = np.asarray(ft, np.float64)
    if mask is not None:
        mask = np.broadcast_to(mask, py.shape); py = py[mask]; ft = ft[mask]
    d = py - ft
    big = np.abs(ft) > max(tiny, 1e-6 * np.nanmax(np.abs(ft)) if ft.size else tiny)
    rel = np.abs(d[big]) / np.abs(ft[big]) if big.any() else np.array([0.0])
    s_ft = np.nansum(ft); s_py = np.nansum(py)
    return dict(n=int(py.size), max_abs=float(np.nanmax(np.abs(d))) if d.size else 0.0,
                max_rel=float(np.nanmax(rel)), p99_rel=float(np.nanpercentile(rel, 99)),
                frac_rel_gt_1e4=float(np.mean(rel > 1e-4)), sum_ratio=float(s_py / s_ft) if s_ft != 0 else np.nan,
                r=float(np.corrcoef(py.ravel(), ft.ravel())[0, 1]) if py.size > 2 and np.std(ft) > 0 else np.nan)

IOAPI_FIXED = ['IOAPI_VERSION', 'EXEC_ID', 'FTYPE', 'CDATE', 'CTIME', 'WDATE', 'WTIME', 'SDATE', 'STIME', 'TSTEP', 'NTHIK',
               'NCOLS', 'NROWS', 'NLAYS', 'NVARS', 'GDTYP', 'P_ALP', 'P_BET', 'P_GAM', 'XCENT', 'YCENT', 'XORIG', 'YORIG',
               'XCELL', 'YCELL', 'VGTYP', 'VGTOP', 'VGLVLS', 'GDNAM', 'UPNAM', 'VAR-LIST', 'FILEDESC', 'HISTORY']

def write_ioapi(path, template, data, *, sdate, stime, tstep, units=None, lon0=None, lat0=None, dlon=None, dlat=None,
                gdnam='SC_BENCH', nlays=1):
    """Write an IOAPI-3 lat-lon file. data: {var: array (T, LAY, ROW, COL) or (T, ROW, COL)}; template supplies attrs."""
    t = attrs(template)
    names = list(data)
    arrs = {k: (np.asarray(v, np.float32)[:, None] if np.asarray(v).ndim == 3 else np.asarray(v, np.float32)) for k, v in data.items()}
    T, L, R, C = arrs[names[0]].shape
    with Dataset(path, 'w', format='NETCDF3_64BIT_OFFSET') as d:
        d.createDimension('TSTEP', None); d.createDimension('DATE-TIME', 2); d.createDimension('LAY', L)
        d.createDimension('VAR', len(names)); d.createDimension('ROW', R); d.createDimension('COL', C)
        g = dict(t)
        g.update(SDATE=np.int32(sdate), STIME=np.int32(stime), TSTEP=np.int32(tstep), NCOLS=np.int32(C), NROWS=np.int32(R),
                 NLAYS=np.int32(L), NVARS=np.int32(len(names)), GDTYP=np.int32(1), P_ALP=0.0, P_BET=0.0, P_GAM=0.0,
                 XCENT=0.0, YCENT=0.0, XORIG=float(lon0), YORIG=float(lat0), XCELL=float(dlon), YCELL=float(dlat),
                 GDNAM=f'{gdnam:<16s}', UPNAM=f'{"BENCH":<16s}', FILEDESC='MEGAN-SC benchmark input',
                 HISTORY='written by megan_sc benchmark',
                 **{'VAR-LIST': ''.join(f'{n:<16s}' for n in names)})
        g['VGLVLS'] = np.array(t['VGLVLS'], dtype=np.float32)[:2] if np.size(t['VGLVLS']) >= 2 else np.array([0, 0], np.float32)
        for k in IOAPI_FIXED:
            if k in g: d.setncattr(k, g[k])
        tf = d.createVariable('TFLAG', 'i4', ('TSTEP', 'VAR', 'DATE-TIME'))
        tf.units = '<YYYYDDD,HHMMSS>'; tf.long_name = 'TFLAG'.ljust(16); tf.var_desc = 'Timestep-valid flags'.ljust(80)
        for k in names:
            v = d.createVariable(k, 'f4', ('TSTEP', 'LAY', 'ROW', 'COL'))
            v.long_name = k.ljust(16); v.units = ((units or {}).get(k, 'none')).ljust(16); v.var_desc = k.ljust(80)
        for ti in range(T):
            if tstep == 0:
                date, time = sdate, stime
            else:
                hh = (stime // 10000) + ti * (tstep // 10000); date, time = sdate + hh // 24, (hh % 24) * 10000
            for vi, k in enumerate(names):
                tf[ti, vi, :] = [date, time]
                d.variables[k][ti] = arrs[k][ti]


def drop_vars(src, dst, drop):
    """Copy an IOAPI file without the variables in ``drop`` (header NVARS, VAR-LIST and TFLAG adjusted).
    Used to make the no-soil-data MGNMET for MEGSEA (SOIM1, SOIT1, SLTYP removed)."""
    with Dataset(src) as s, Dataset(dst, 'w', format=s.data_model) as d:
        keep = [v for v in s.variables if v != 'TFLAG' and v not in drop]
        for n, dim in s.dimensions.items():
            d.createDimension(n, None if dim.isunlimited() else (len(keep) if n == 'VAR' else len(dim)))
        for k in s.ncattrs():
            v = s.getncattr(k)
            if k == 'NVARS': v = np.int32(len(keep))
            if k == 'VAR-LIST': v = ''.join(f'{n:<16s}' for n in keep)
            d.setncattr(k, v)
        idx = [list(s.variables)[1:].index(n) for n in keep]
        for n in ['TFLAG'] + keep:
            sv = s.variables[n]; dv = d.createVariable(n, sv.dtype, sv.dimensions)
            dv.setncatts({a: sv.getncattr(a) for a in sv.ncattrs()})
            dv[:] = sv[:][:, idx, :] if n == 'TFLAG' else sv[:]
''',
 'bench_se_us.py': r'''"""Stage-by-stage benchmark of the MEGAN-SC Python port against the official MEGANv3.21 Fortran
(Guenther 2024, Zenodo 10.5281/zenodo.10526206), on the official SE_US test case (2018-05-02).
usage: python bench_se_us.py <megan_sc src dir> <MEGANv3.21 dir> <fortran out dir> <out csv> [GAMSM_YN] [GAMBD_YN]"""
import sys, os, numpy as np, pandas as pd
SRC, MGN, FD, OUTCSV = sys.argv[1:5]; SM = (sys.argv[5] if len(sys.argv) > 5 else 'N') == 'Y'; BD = (sys.argv[6] if len(sys.argv) > 6 else 'N') == 'Y'
sys.path.insert(0, SRC); sys.path.insert(0, os.path.dirname(__file__))
from megan_sc.engine import canopy, megvea, daymet, species as sp
PATCH = os.environ.get('MEGAN_PATCH', '0') == '1'
if PATCH:
    from megan_sc import fixes; print('engine patches:', fixes.apply_engine_patches())
from ioapi import rd, compare
INP = f'{MGN}/Input'; MET = f'{INP}/MGNMET/MET.MEGAN.SE_US.rad45.2018122.ncf'; LAIF = f'{INP}/MAP/LAI3_SE_US.ncf'
JJJ = 122; NT = 24
lat = rd(LAIF, 'LAT')[0]; lon = rd(LAIF, 'LONG')[0]
lai_c = rd(LAIF, f'LAI{int(np.ceil(JJJ / 8.0)):02d}')[0]; lai_p = rd(LAIF, f'LAI{int(np.ceil(JJJ / 8.0)) - 1:02d}')[0]   # findlai.f, NLAI=46
cts = rd(f'{MGN}/work/CANTYPFILE', 'CTS')                       # (6, ROW, COL), percent
met = {k: rd(MET, k)[:NT] for k in ('TEMP2', 'PAR', 'WINDSPD', 'PRES', 'QV')}
rows = []
def add(stage, var, st): rows.append(dict(stage=stage, var=var, **st))
# ---- S1 DAYMET
# daymet.f averages AVEBY = MIN(24, MXREC) records; the official run script (RLENG = 230000) gives 23 (00-22 UTC).
# MinT is not compared: daymet.f allocates TEMP(24,...) and takes MINVAL over all 24 rows, so with 23 records read
# MinT = 0 (unused unless GAMLT_YN = Y).
NAVG = int(os.environ.get('DAYMET_NREC', '23'))
dm = daymet.daily_aggregate(met['TEMP2'][:NAVG], met['PAR'][:NAVG], met['WINDSPD'][:NAVG], axis=0)
for k in ('D_TEMP', 'D_PPFD', 'MaxT', 'MaxWS'):
    add('S1 daymet', k, compare(dm[k], rd(f'{FD}/DAYMET.ncf', k)[0]))
# ---- S2 MEGCAN (Python canopy model vs CANMET), hour by hour
CANV = [('SunleafTK', 'SunT'), ('ShadeleafTK', 'ShaT'), ('SunPPFD', 'SunP'), ('ShadePPFD', 'ShaP'), ('SunFrac', 'SunF')]
can_py = {}; acc = {f: ([], []) for f, _ in CANV}
CANF = {f: rd(f'{FD}/CANMET.ncf', f, squeeze_lay=False).astype(np.float32) for f, _ in CANV}   # (T, LAY, ROW, COL)
ERSF = {n: rd(f'{FD}/MGNERS.ncf', n).astype(np.float32) for n in sp.NAMES}
veg = (lai_c > 0) & (cts.sum(0) > 0)
for t in range(NT):
    c = canopy.megcan(met['TEMP2'][t], met['PAR'][t], met['WINDSPD'][t], met['PRES'][t], met['QV'][t],
                      lat, lon, lai_c, cts, JJJ, float(t))
    can_py[t] = c
    for f, p in CANV:
        acc[f][0].append(np.moveaxis(c[p], -1, 0).astype(np.float32)); acc[f][1].append(CANF[f][t])
for f, _ in CANV:
    add('S2 megcan', f, compare(np.stack(acc[f][0]), np.stack(acc[f][1]), mask=np.broadcast_to(veg, np.stack(acc[f][1]).shape)))
# ---- S3 MEGVEA isolated (Fortran CANMET + DAYMET fed to Python megvea) and S4 end-to-end (Python megcan -> Python megvea)
ldf = np.zeros((19,) + lat.shape)
for s in sp.LDF_FROM_MAP: ldf[s] = rd(f'{INP}/MAP/LDF_SE_US.2019b.ncf', f'LDF{s + 1:02d}')[0]
dT = rd(f'{FD}/DAYMET.ncf', 'D_TEMP')[0]; dP = rd(f'{FD}/DAYMET.ncf', 'D_PPFD')[0]
smf = os.environ.get('SMFILE', f'{MGN}/Output/INT/MGNSEA.SE_US.2018122.ncf'); GSM = rd(smf, 'GAMSM') if SM else None
er3 = []; er4 = []; erf = []
for t in range(NT):
    canf = {p: np.moveaxis(CANF[f][t].astype(np.float64), 0, -1) for f, p in CANV}
    gsm = GSM[t] if SM else None
    kw = dict(laic=lai_c, laip=lai_p, d_temp=dT, d_ppfd=dP, gamsm=gsm, ldf_map=ldf, nlai=46, apply_co2=False, apply_bidir=BD)
    er3.append(megvea.emission_activity(canf, **kw).astype(np.float32))
    er4.append(megvea.emission_activity(can_py[t], **kw).astype(np.float32))
    erf.append(np.stack([ERSF[n][t] for n in sp.NAMES]))
    print('hour', t, flush=True)
er3, er4, erf = np.stack(er3), np.stack(er4), np.stack(erf)        # (T, 19, ROW, COL)
for s, n in enumerate(sp.NAMES):
    if n == 'NO':
        continue                                                   # soil NO is handled by MEGSEA (BDSNP in the reference)
    add('S3 megvea (Fortran canopy in)', n, compare(er3[:, s], erf[:, s]))
    add('S4 end-to-end ER', n, compare(er4[:, s], erf[:, s]))
d = pd.DataFrame(rows); d.to_csv(OUTCSV, index=False)
pd.set_option('display.width', 220)
print(d.groupby('stage').agg(max_rel=('max_rel', 'max'), p99_rel=('p99_rel', 'max'), worst_sum_ratio_dev=('sum_ratio', lambda x: float(np.nanmax(np.abs(x - 1))))).to_string())
print(d.to_string(float_format=lambda v: f'{v:.3g}'))
''',
 'bench_soil.py': r'''"""Soil NO (Yienger-Levy path) and GAMSM: Python port vs official MEGSEA (BDSNP_YN=N), SE_US 2018-05-02.
usage: bench_soil.py <out csv>; env ENGINE_SRC (megan_sc src), MGN (MEGANv3.21 dir), FD (Fortran output dir), MEGAN_PATCH=0|1"""
import sys, os, numpy as np, pandas as pd
sys.path.insert(0, os.environ.get('ENGINE_SRC', os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', 'src'))); sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from megan_sc.engine import soil
if os.environ.get('MEGAN_PATCH', '0') == '1':
    from megan_sc import fixes; print('engine patches:', fixes.apply_engine_patches())
from ioapi import rd, compare
M = os.environ.get('MGN', os.path.expanduser('~/scratch/megan321/MEGANv3.21')); FD = os.environ.get('FD', 'se_us/fort'); MET = f'{M}/Input/MGNMET/MET.MEGAN.SE_US.rad45.2018122.ncf'
L = f'{M}/Input/MAP/LAI3_SE_US.ncf'; lat = rd(L, 'LAT')[0]; lai = rd(L, 'LAI16')[0]; cts = rd(f'{M}/work/CANTYPFILE', 'CTS')
D = 2018122; rows = []
gday, _ = soil.growseason(D, lat, corrected=False)
for path in ('soil', 'nosoil'):
    ref = rd(f'{FD}/MGNSEA_YL_{path}.ncf', 'GAMNO'); py = [dict() for _ in range(24)]
    for t in range(24):
        T = rd(MET, 'TEMP2', t); pa = rd(MET, 'PREC_ADJ', t)
        kw = dict(soilt=rd(MET, 'SOIT1', t), soilm=rd(MET, 'SOIM1', t), soiltype=np.rint(rd(MET, 'SLTYP', t)).astype(int)) if path == 'soil' else {}
        for corr in (False, True):
            cfno, cfnog = soil.soilnox(T, lai, lat, D, precadj=pa, corrected=corr, **kw)
            g = soil.gamma_no(cfno, cfnog, cts, soil.growseason(D, lat, corrected=corr)[0])
            py[t][corr] = g
    for corr in (False, True):
        rows.append(dict(test=f'GAMNO YL {path}-path, corrected={corr}', **compare(np.stack([p[corr] for p in py]), ref)))
gsm_ref = rd(f'{FD}/MGNSEA_YL_soil.ncf', 'GAMSM')
gsm_py = np.stack([soil.gamma_sm(rd(MET, 'SOIM1', t), soil.wilting_point(np.rint(rd(MET, 'SLTYP', t)).astype(int))) for t in range(24)])
rows.append(dict(test='GAMSM (WWLT by soil type)', **compare(gsm_py, gsm_ref)))
d = pd.DataFrame(rows); print(d.to_string(float_format=lambda v: f'{v:.3g}')); d.to_csv(sys.argv[1] if len(sys.argv) > 1 else 'se_us/bench_soil.csv', index=False)
''',
 'bench_spec.py': r'''"""Speciation + EF application + unit chain: Python (speciation.py) vs official mgn2mech (RUN_CONVERSION=N, SPCTONHR=Y).
usage: bench_spec.py <out csv>; env ENGINE_SRC, MGN, FD as in bench_soil.py"""
import sys, os, numpy as np, pandas as pd
sys.path.insert(0, os.environ.get('ENGINE_SRC', os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', 'src'))); sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from megan_sc.engine import speciation as spc, species as sp
from ioapi import rd
from netCDF4 import Dataset
M = os.environ.get('MGN', os.path.expanduser('~/scratch/megan321/MEGANv3.21')); FD = os.environ.get('FD', 'se_us/fort')
EF = np.stack([rd(f'{M}/Input/MAP/EFMAP.2019b.SE_US.ncf', f'EF_{n}')[0] for n in sp.NAMES])
AREA = 9000.0 * 9000.0
out = Dataset(f'{FD}/MGN201_tonhr.ncf'); names = spc.SPCA_NAMES; fvars = [k for k in out.variables][1:]; assert all(a[:16] == b for a, b in zip(names, fvars))
ERS = {n: rd(f'{FD}/MGNERS.ncf', n) for n in sp.NAMES}; GNO = rd(f'{FD}/MGNSEA_YL_nosoil.ncf', 'GAMNO')
sum_py = np.zeros(len(names)); sum_ft = np.zeros(len(names)); maxrel = np.zeros(len(names))
for t in range(24):
    er = np.stack([ERS[n][t] for n in sp.NAMES])
    fl = spc.speciate_to_compounds(er, EF, gamno=GNO[t])
    ton = spc.compound_tonnes(spc.flux_to_mol_s(fl, AREA), 3600.0)
    for i, n in enumerate(names):
        f = np.asarray(out[fvars[i]][t, 0], np.float64); p = ton[i]
        sum_py[i] += p.sum(); sum_ft[i] += f.sum()
        big = np.abs(f) > 1e-6 * max(np.abs(f).max(), 1e-30)
        if big.any(): maxrel[i] = max(maxrel[i], float(np.max(np.abs(p[big] - f[big]) / np.abs(f[big]))))
d = pd.DataFrame(dict(species=names, parent=[sp.NAMES[c] if c >= 0 else '-' for c in spc.SPEC_CLASS], tonnes_py=sum_py, tonnes_fortran=sum_ft, max_rel_cellhour=maxrel))
d['ratio'] = d.tonnes_py / d.tonnes_fortran.replace(0, np.nan)
d.to_csv(sys.argv[1] if len(sys.argv) > 1 else 'se_us/bench_speciation.csv', index=False)
cl = d.groupby('parent')[['tonnes_py', 'tonnes_fortran']].sum(); cl['ratio'] = cl.tonnes_py / cl.tonnes_fortran
print(cl.to_string(float_format=lambda v: f'{v:.6g}'))
print('species with |ratio-1|>1e-5:', d[(d.ratio - 1).abs() > 1e-5][['species', 'parent', 'tonnes_py', 'tonnes_fortran', 'ratio']].to_string())
print('max cell-hour rel diff over all species: %.3g' % d.max_rel_cellhour.max(), '| species zero in both:', int(((d.tonnes_py == 0) & (d.tonnes_fortran == 0)).sum()))
''',
 'make_sc_inputs.py': r'''"""Build MEGANv3.21 IOAPI inputs on the Santa Catarina analysis grid (0.025 deg, 144 x 224, south-first)
for one day, from the 2019 inputs of a run folder (MODIS LAI with Earth Engine default resampling, MapBiomas growth
forms, ERA5-Land monthly means).
The diurnal cycle is idealised (T: cosine, amplitude 5 K, max 15 h local; PAR: clear-sky cos(zenith)
shape scaled to the monthly-mean PAR).  These are TEST inputs for a code-equivalence benchmark, not weather.
usage: make_sc_inputs.py <engine src> <megan_sc_outputs dir> <MEGANv3.21 dir> <outdir> <month> <day>"""
import sys, os, datetime, numpy as np, rasterio
SRC, RAW, MGN, OUT, MON, DAY = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], int(sys.argv[5]), int(sys.argv[6])
sys.path.insert(0, SRC); sys.path.insert(0, os.path.dirname(__file__))
from megan_sc import landcover, fixes
from ioapi import write_ioapi
os.makedirs(OUT, exist_ok=True)
YEAR = 2019; RES = 0.025
doy = datetime.date(YEAR, MON, DAY).timetuple().tm_yday; SDATE = YEAR * 1000 + doy
def tif(path, band=None):
    with rasterio.open(path) as r:
        a = r.read().astype(np.float64); tr = r.transform
    return np.flip(a, axis=1), tr                                   # south-first rows
lai12 = np.stack([np.nan_to_num(tif(f'{RAW}/{YEAR}/inputs/lai_{YEAR}_{m:02d}.tif')[0][0], nan=0.0) for m in range(1, 13)])
met, tr = tif(f'{RAW}/{YEAR}/inputs/met_{YEAR}_{MON:02d}.tif')    # TEMP2, PAR, WINDSPD, PRES, QV, SOILM (monthly means)
ny, nx = lai12.shape[1:]
lon = tr.c + RES * (np.arange(nx) + 0.5); lat = (tr.f + tr.e * ny) + RES * (np.arange(ny) + 0.5)   # south-first centres
LON, LAT = np.meshgrid(lon, lat)
gf = dict(np.load(f'{RAW}/{YEAR}/inputs/growthform_{YEAR}.npz'))
ctf = landcover.canopy_type_fractions(gf, 3)                       # percent, (6, ny, nx); tree -> type 3 (production setting)
m = {'temp': met[0], 'par': met[1], 'wind': met[2], 'pres': met[3], 'qv': met[4], 'soilm': met[5]}
m, bad = fixes.fill_met_nearest(m)                                  # ocean / no-data cells -> nearest land (FIX-1 rule)
hours = np.arange(24)
loc = (hours[:, None, None] + LON[None] / 15.0) % 24.0
T = m['temp'][None] + 5.0 * np.cos(2 * np.pi * (loc - 15.0) / 24.0)
decl = np.deg2rad(23.45) * np.sin(2 * np.pi * (284 + doy) / 365.0)
cosz = np.sin(np.deg2rad(LAT))[None] * np.sin(decl) + np.cos(np.deg2rad(LAT))[None] * np.cos(decl) * np.cos(np.deg2rad(15.0 * (loc - 12.0)))
shape = np.maximum(cosz, 0.0)
PAR = shape * (m['par'] * 24.0 / np.maximum(shape.sum(0), 1e-6))[None]
const = lambda a: np.repeat(a[None], 24, axis=0)
common = dict(lon0=lon[0] - RES / 2, lat0=lat[0] - RES / 2, dlon=RES, dlat=RES)
TPL = f'{MGN}/Input/MGNMET/MET.MEGAN.SE_US.rad45.2018122.ncf'
write_ioapi(f'{OUT}/MET.ncf', TPL, {'TEMP2': T, 'PAR': PAR, 'WINDSPD': const(m['wind']), 'PRES': const(m['pres']), 'QV': const(m['qv']), 'PREC_ADJ': np.ones_like(T)},
            sdate=SDATE, stime=0, tstep=10000, units={'TEMP2': 'K', 'PAR': 'WATTS/M**2', 'WINDSPD': 'M/S', 'PRES': 'Pa', 'QV': 'KG/KG'}, **common)
laid = {f'LAI{k + 1:02d}': lai12[k][None] for k in range(12)}
# megvea.f builds the MGNERS header from leftover IOAPI description arrays and expects >= 20 variables in the
# LAI file (as in the 46-period official LAI3 file).  Pad with unused LAI13..LAI46 (never read with NLAI = 12).
laid.update({f'LAI{k + 1:02d}': np.zeros((1,) + lai12.shape[1:]) for k in range(12, 46)})
laid['LAT'] = LAT[None]; laid['LONG'] = LON[None]
write_ioapi(f'{OUT}/LAI.ncf', f'{MGN}/Input/MAP/LAI3_SE_US.ncf', laid, sdate=0, stime=0, tstep=10000, **common)
write_ioapi(f'{OUT}/CANTYP.ncf', f'{MGN}/Input/MAP/LDF_SE_US.2019b.ncf', {'CTS': ctf}, sdate=0, stime=0, tstep=10000, **common)
ldf = {'LDF03': 0.6, 'LDF04': 0.9, 'LDF05': 0.2, 'LDF06': 0.4}     # MEGVEA.EXT constants used by the production run
write_ioapi(f'{OUT}/LDF.ncf', f'{MGN}/Input/MAP/LDF_SE_US.2019b.ncf', {k: np.full((1, ny, nx), v) for k, v in ldf.items()}, sdate=0, stime=0, tstep=10000, **common)
with open(f'{OUT}/GRIDDESC', 'w') as f:
    f.write("' '\n'LATLON'\n  1  0.0  0.0  0.0  0.0  0.0\n' '\n'SC_BENCH'\n")
    f.write(f"'LATLON'  {lon[0] - RES / 2:.6f}  {lat[0] - RES / 2:.6f}  {RES:.6f}  {RES:.6f}  {nx}  {ny}  1\n' '\n")
np.savez_compressed(f'{OUT}/py_inputs.npz', T=T, PAR=PAR, wind=m['wind'], pres=m['pres'], qv=m['qv'], lai12=lai12, ctf=ctf, LAT=LAT, LON=LON, doy=doy, sdate=SDATE)
print('SC benchmark inputs:', OUT, 'SDATE', SDATE, 'grid', ny, nx, '| met cells filled', int(bad.sum()), '| LAI>0 cells', int((lai12[MON - 1] > 0).sum()))
''',
 'bench_sc.py': r'''"""Python port vs official MEGANv3.21 Fortran on the Santa Catarina grid (Southern Hemisphere, 0.025 deg).
usage: bench_sc.py <engine src (with fixes.py)> <MEGAN-SC repo dir> <case dir> <out csv>"""
import sys, os, numpy as np, pandas as pd
SRC, REPO, D, OUTCSV = sys.argv[1:5]
sys.path.insert(0, SRC); sys.path.insert(0, os.path.dirname(__file__))
from megan_sc.engine import canopy, megvea, daymet, soil, gamma, species as sp, speciation as spc
from megan_sc import fixes, run, domain as dom
from ioapi import rd, compare
Z = np.load(f'{D}/py_inputs.npz'); F = f'{D}/fort'
T, PAR = Z['T'], Z['PAR']; LAT, LON = Z['LAT'], Z['LON']; ctf = Z['ctf']; lai12 = Z['lai12']; doy = int(Z['doy']); SD = int(Z['sdate'])
mon = int(pd.Timestamp(f'{SD // 1000}-01-01').dayofyear) and (pd.Timestamp(f'{SD // 1000}-01-01') + pd.Timedelta(days=doy - 1)).month
laic = lai12[mon - 1]; laip = lai12[0] if mon == 1 else lai12[mon - 2]   # findlai.f with NLAI = 12 and MXLAI = 1 (January -> LAI01)
wind, pres, qv = Z['wind'], Z['pres'], Z['qv']
veg = (laic > 0) & (ctf.sum(0) > 0)
rows = []
def add(stage, var, st, **kw): rows.append(dict(case=os.path.basename(D), stage=stage, var=var, **st, **kw))
dm = daymet.daily_aggregate(T, PAR, np.repeat(wind[None], 24, 0), axis=0)
for k in ('D_TEMP', 'D_PPFD', 'MaxT'):
    add('S1 daymet', k, compare(dm[k], rd(f'{F}/DAYMET.ncf', k)[0]))
CANV = [('SunleafTK', 'SunT'), ('ShadeleafTK', 'ShaT'), ('SunPPFD', 'SunP'), ('ShadePPFD', 'ShaP'), ('SunFrac', 'SunF')]
CANF = {f: rd(f'{F}/CANMET.ncf', f, squeeze_lay=False).astype(np.float32) for f, _ in CANV}
ERF = {n: rd(f'{F}/MGNERS.ncf', n).astype(np.float32) for n in sp.NAMES}
dT = rd(f'{F}/DAYMET.ncf', 'D_TEMP')[0]; dP = rd(f'{F}/DAYMET.ncf', 'D_PPFD')[0]
canacc = {f: [] for f, _ in CANV}; erp = []; eru = []; erf = []
for t in range(24):
    c = canopy.megcan(T[t], PAR[t], wind, pres, qv, LAT, LON, laic, ctf, doy, float(t))
    for f, p in CANV: canacc[f].append(np.moveaxis(c[p], -1, 0).astype(np.float32))
    kw = dict(laic=laic, laip=laip, d_temp=dT, d_ppfd=dP, gamsm=None, nlai=12, apply_co2=False, apply_bidir=True)
    fixes.apply_engine_patches(); erp.append(megvea.emission_activity(c, **kw).astype(np.float32))
    gamma.gamtp = fixes._ORIG['gamtp']; eru.append(megvea.emission_activity(c, **kw).astype(np.float32)); fixes.apply_engine_patches()
    erf.append(np.stack([ERF[n][t] for n in sp.NAMES]))
for f, _ in CANV:
    a = np.stack(canacc[f]); add('S2 megcan', f, compare(a, CANF[f], mask=np.broadcast_to(veg, a.shape)))
erp, eru, erf = np.stack(erp), np.stack(eru), np.stack(erf)
# emission-weighted effect sizes: production EF grid (MapBiomas growth forms x MEGAN v3 ecotypes)
_cp = f'{REPO}/config/example_config.yaml'
cfg = run.load_config(_cp if os.path.exists(_cp) else f'{REPO}/example_config.yaml'); cfg['domain']['mode'] = 'bbox'
for k in ('ef_table', 'speciation_dir', 'ecotype'):
    v = cfg['datasets']['emission_factors'].get(k)
    if v and not os.path.isabs(v): cfg['datasets']['emission_factors'][k] = os.path.join(REPO, v)
domn = dom.build_domain(cfg)
gf = {k: v for k, v in np.load(os.path.join(os.environ['RAW'], '2019/inputs/growthform_2019.npz')).items()}
ef = run.build_ef_grid_multiform(cfg, domn, gf); area = domn['area']
assert ef.shape[1:] == laic.shape, (ef.shape, laic.shape)
def day_tonnes(er):            # tonnes per class for the day (hourly steps), NO excluded (soil path)
    out = np.zeros(19)
    for t in range(24):
        fl = spc.speciate_to_compounds(er[t].astype(np.float64), ef, gamno=np.zeros(laic.shape))
        out += spc.compounds_to_classes(spc.compound_tonnes(spc.flux_to_mol_s(fl, area), 3600.0)).sum(axis=(1, 2))
    return out
tp, tu, tf = day_tonnes(erp), day_tonnes(eru), day_tonnes(erf)
for s, n in enumerate(sp.NAMES):
    if n == 'NO':
        continue
    add('S3+S4 ER, patched engine (FIX-6)', n, compare(erp[:, s], erf[:, s]), day_tonnes_ratio=tp[s] / tf[s])
    add('S4 ER, original port (no FIX-6)', n, compare(eru[:, s], erf[:, s]), day_tonnes_ratio=tu[s] / tf[s])
# soil NO, air-temperature path (as in production); Fortran growseason with its SH branch
gref = rd(f'{F}/MGNSEA.ncf', 'GAMNO')
for patched in (True, False):
    if patched: fixes.apply_engine_patches()
    else: soil.soilnox = fixes._ORIG['soilnox']
    for corr in (False, True):
        g = []
        for t in range(24):
            cfno, cfnog = soil.soilnox(T[t], laic, LAT, SD, corrected=corr)
            g.append(soil.gamma_no(cfno, cfnog, ctf, soil.growseason(SD, LAT, corrected=corr)[0]))
        add(f'soil NO GAMNO ({"FIX-7" if patched else "original port"}, SH season {"corrected" if corr else "as Fortran"})', 'NO', compare(np.stack(g), gref))
fixes.apply_engine_patches()
d = pd.DataFrame(rows); d.to_csv(OUTCSV, index=False)
pd.set_option('display.width', 250)
print(d.to_string(float_format=lambda v: f'{v:.3g}'))
''',
 'run_fortran_se_us.sh': r'''#!/bin/bash
# Runs the official MEGANv3.21 binaries (as shipped in Zenodo 10.5281/zenodo.10526206) on the SE_US test case.
# Usage: run_fortran.sh <outdir> [GAMSM_YN] [GAMBD_YN] [NLAI]
set -e
M=${MGN:-$HOME/scratch/megan321/MEGANv3.21}; OUT=$1; SM=${2:-N}; BD=${3:-N}; NL=${4:-46}
export LD_LIBRARY_PATH=${OMP_LIB:-$HOME/.local/lib}:$LD_LIBRARY_PATH PROMPTFLAG=N EXECUTION_ID=bench
mkdir -p $OUT; INP=$M/Input; JD=2018122
export GRIDDESC=$INP/MAP/GRIDDESC GDNAM3D=SE_US
# DAYMET
( export EPISDATE=$JD STIME=0 RLENG=230000 NDAYS=1 MGNMET001=$INP/MGNMET/MET.MEGAN.SE_US.rad45.$JD.ncf DailyMET=$OUT/DAYMET.ncf
  rm -f $DailyMET; $M/src/DAYMET/daymet > $OUT/log_daymet.txt 2>&1 )
# MEGCAN
( export NLAI=$NL SDATE=$JD STIME=0 RLENG=240000 CANTYPFILE=$M/work/CANTYPFILE LAIFILE=$INP/MAP/LAI3_SE_US.ncf \
         MGNMET=$INP/MGNMET/MET.MEGAN.SE_US.rad45.$JD.ncf CANMET=$OUT/CANMET.ncf
  rm -f $CANMET; $M/src/MEGCAN/megcan > $OUT/log_megcan.txt 2>&1 )
# MEGVEA
( export Layers=5 NLAI=$NL GAMAQ_YN=N GAMCO2_YN=N GAMHW_YN=N GAMHT_YN=N GAMLT_YN=N GAMSM_YN=$SM GAMBD_YN=$BD \
         SDATE=$JD STIME=0 RLENG=240000 N_MaxT=1 N_MinT=1 N_MaxWS=1 LAIFILE=$INP/MAP/LAI3_SE_US.ncf CANMET=$OUT/CANMET.ncf \
         DailyMET=$OUT/DAYMET.ncf SMFILE=${SMFILE:-$M/Output/INT/MGNSEA.SE_US.$JD.ncf} AQFILE=$INP/MAP/W126_SE_US.ncf \
         LDFILE=$INP/MAP/LDF_SE_US.2019b.ncf MGNERS=$OUT/MGNERS.ncf
  rm -f $MGNERS; $M/src/MEGVEA/megvea > $OUT/log_megvea.txt 2>&1 )
echo "fortran done -> $OUT"; tail -3 $OUT/log_megvea.txt
''',
 'run_fortran_se_us_soil_spec.sh': r'''#!/bin/bash
# Official MEGSEA (Yienger-Levy soil NO, BDSNP_YN=N; with and without soil data) and mgn2mech (RUN_CONVERSION=N,
# SPCTONHR=Y: 201 MEGAN species in tonnes/hr) on the SE_US test case. Needs MGNERS.ncf from run_fortran_se_us.sh in <outdir>.
# usage: run_fortran_se_us_soil_spec.sh <outdir>
set -e
M=${MGN:-$HOME/scratch/megan321/MEGANv3.21}; OUT=$(cd $1 && pwd); HERE=$(cd $(dirname $0) && pwd)
export LD_LIBRARY_PATH=${OMP_LIB:-$HOME/.local/lib}:$LD_LIBRARY_PATH PROMPTFLAG=N EXECUTION_ID=bench
INP=$M/Input; JD=2018122; MET=$INP/MGNMET/MET.MEGAN.SE_US.rad45.$JD.ncf
export GRIDDESC=$INP/MAP/GRIDDESC GDNAM3D=SE_US SDATE=$JD STIME=0 RLENG=240000
python3 -c "import sys; sys.path.insert(0, '$HERE'); from ioapi import drop_vars; drop_vars('$MET', '$OUT/MET_nosoil.ncf', ('SOIM1', 'SOIT1', 'SLTYP'))"
for path in soil nosoil; do
  MM=$MET; [ $path = nosoil ] && MM=$OUT/MET_nosoil.ncf
  ( export NLAI=46 BDSNP_YN=N EPIC=N CANTYPFILE=$M/work/CANTYPFILE LAIFILE=$INP/MAP/LAI3_SE_US.ncf MGNMET=$MM MGNSEA=$OUT/MGNSEA_YL_$path.ncf
    rm -f $MGNSEA; $M/src/MEGSEA/megsea > $OUT/log_megsea_$path.txt 2>&1 )
done
( export TSTEP=10000 RUN_SPECIATE=Y RUN_CONVERSION=N SPCTONHR=Y BDSNP_YN=N MECHANISM=CB6X EFMAPS=$INP/MAP/EFMAP.2019b.SE_US.ncf \
         MGNERS=$OUT/MGNERS.ncf MGNSEA=$OUT/MGNSEA_YL_nosoil.ncf MGNOUT=$OUT/MGN201_tonhr.ncf
  rm -f $MGNOUT; $M/src/MGN2MECH/mgn2mech > $OUT/log_mgn2mech.txt 2>&1 )
for p in megsea_soil megsea_nosoil mgn2mech; do printf "%s: " $p; grep -c "Normal Completion" $OUT/log_$p.txt || true; done
''',
 'run_fortran_sc.sh': r'''#!/bin/bash
# Official MEGANv3.21 binaries on the Santa Catarina benchmark inputs (lat-lon IOAPI). usage: run_fortran_sc.sh <dir>
set -e
M=${MGN:-$HOME/scratch/megan321/MEGANv3.21}; D=$(cd $1 && pwd); SD=$(python3 -c "import numpy as np;print(int(np.load('$D/py_inputs.npz')['sdate']))")
export LD_LIBRARY_PATH=${OMP_LIB:-$HOME/.local/lib}:$LD_LIBRARY_PATH PROMPTFLAG=N EXECUTION_ID=bench GRIDDESC=$D/GRIDDESC GDNAM3D=SC_BENCH
mkdir -p $D/fort; O=$D/fort
( export EPISDATE=$SD STIME=0 RLENG=240000 NDAYS=1 MGNMET001=$D/MET.ncf DailyMET=$O/DAYMET.ncf; rm -f $DailyMET; $M/src/DAYMET/daymet > $O/log_daymet.txt 2>&1 )
( export NLAI=12 SDATE=$SD STIME=0 RLENG=240000 CANTYPFILE=$D/CANTYP.ncf LAIFILE=$D/LAI.ncf MGNMET=$D/MET.ncf CANMET=$O/CANMET.ncf; rm -f $CANMET; $M/src/MEGCAN/megcan > $O/log_megcan.txt 2>&1 )
( export NLAI=12 SDATE=$SD STIME=0 RLENG=240000 BDSNP_YN=N EPIC=N CANTYPFILE=$D/CANTYP.ncf LAIFILE=$D/LAI.ncf MGNMET=$D/MET.ncf MGNSEA=$O/MGNSEA.ncf; rm -f $MGNSEA; $M/src/MEGSEA/megsea > $O/log_megsea.txt 2>&1 )
( export Layers=5 NLAI=12 GAMAQ_YN=N GAMCO2_YN=N GAMHW_YN=N GAMHT_YN=N GAMLT_YN=N GAMSM_YN=N GAMBD_YN=Y SDATE=$SD STIME=0 RLENG=240000 \
         N_MaxT=1 N_MinT=1 N_MaxWS=1 LAIFILE=$D/LAI.ncf CANMET=$O/CANMET.ncf DailyMET=$O/DAYMET.ncf SMFILE=$O/MGNSEA.ncf \
         LDFILE=$D/LDF.ncf MGNERS=$O/MGNERS.ncf; rm -f $MGNERS; $M/src/MEGVEA/megvea > $O/log_megvea.txt 2>&1 )
for p in daymet megcan megsea megvea; do printf "%s: " $p; grep -c "Normal Completion" $O/log_$p.txt; done
''',
}
for k, v in FILES.items():
    with open(os.path.join(PY, k), 'w') as f: f.write(v)
    if k.endswith('.sh'): os.chmod(os.path.join(PY, k), 0o755)
FIXES_PY = r'''"""megan_sc.fixes -- corrections to the MEGAN-SC production run (adversarial review, Oct 2026).

Copy this file to  megan-sc/src/megan_sc/fixes.py .  It does not modify any existing module; the
corrected production cells (colab_cells_v10.py) call `run_timestep_fixed` instead of
`run.run_timestep`.  Every change is traceable to a MEGANv3.21 source line or to a diagnosed
artefact in the delivered 2003-2025 output.

FIX-1  Coastal / no-data meteorology (CRITICAL).
       ERA5-Land is land-only.  At cells it does not cover, Earth Engine returns a non-finite or a
       very large fill value.  `run.run_timestep` sanitised these with nan_to_num (PAR -> 0) and a
       clip of temperature to [150, 340] K, assuming such cells carry LAI = 0.  In the delivered
       run, 120 coastal cells inside Santa Catarina (821 km2) carry MODIS LAI and MapBiomas
       vegetation, so they were run every hour with T = 340 K and PAR = 0: ISOP, MBO and CO (LDF = 1)
       were exactly 0 and the light-independent part of every other class was multiplied by
       exp(beta*(340-303.15)) (x40 for monoterpenes, x525 for sesquiterpenes).  This produced
       16-22 % of the state BVOC total and ~75 % of the sesquiterpenes in every year.
       Fix: replace every invalid met value by the value of the nearest valid cell (all bands taken
       from the same donor cell) BEFORE the engine runs, and refuse to run (`strict=True`) if a
       vegetated cell is still invalid.
FIX-2  Daily-mean meteorology (MAJOR).
       `run.run_timestep` defaults to D_TEMP = T(hour) - 2 K and D_PPFD = 0.5*PPFD(hour) when the
       caller passes none; the production notebook passed none.  In MEGANv3.21 D_TEMP and D_PPFD
       are the 24-h means of the simulation day (daymet.f L287-320; read by megvea.f L515-523) and
       D_TEMP enters GAMTLD through Eopt = Cleo*exp(0.1*(T24-297)) and Topt (megvea.f GAMTLD).
       Fix: `run_timestep_fixed` REQUIRES d_temp and d_ppfd; the Colab cells compute D_TEMP as the
       mean of the 24 hourly ERA5-Land 2 m temperatures of the UTC day and D_PPFD from the day's PAR.
FIX-3  Leaf age (MODERATE).
       `run.run_timestep` called megvea with LAIp = LAIc and NLAI = 46, which pins GAMMA_A to its
       steady-state branch.  MEGANv3.21 reads the previous LAI period (megvea.f L585-600, FINDLAI).
       Fix: pass the previous month's LAI and NLAI = 12 (TSTLEN = 30 d).
FIX-5  Ecotype no-data on coastal land (MINOR).
       The MEGAN v3 ecotype map (EVT) carries the no-data code 65535 on a thin coastal strip.  Vegetated
       analysis cells covered only by no-data get EF = 0 and emit nothing (28 SC cells, 192 km2,
       ~0.15 % of state BVOC; e.g. the whole municipality of Governador Celso Ramos).
       Fix: `fill_ecotype_nearest` writes a copy of the ecotype file in which no-data pixels take the
       nearest valid ecotype; point the config's emission_factors.ecotype to that copy.
FIX-4  MODIS LAI monthly gaps (MINOR).
       Vegetated cells with no good-quality MCD15A3H retrieval in a month received LAI = 0 and
       therefore zero emission (0-1.4 % of annual isoprene, concentrated in cloudy summer months).
       Fix: `fill_lai_gaps` fills such cell-months from the adjacent valid months.
FIX-6  Canopy-depth factor in GAMTP (MAJOR for fidelity; found by the Fortran benchmark, Oct 2026).
       megvea.f (L725-731) computes
           Ea1L(K) = CDEA(K)*GAMTLD(SunT)*GAMP(SunP)*SunF + GAMTLD(ShaT)*GAMP(ShaP)*(1-SunF)
       i.e. by Fortran operator precedence CDEA multiplies ONLY the sunlit term.  The Python port
       (gamma.gamtp) multiplied both terms by CDEA.  On the official SE_US test case this changed the
       light-dependent ER by up to 25 % per cell-hour (+2 % domain total).  Fix: `gamtp_megan321`
       reproduces the reference code exactly (benchmark: max relative difference 4e-6).
FIX-7  Soil-NO air-temperature path (MODERATE; found by the Fortran benchmark).
       soilnox.F caps TAIR at 303 K BEFORE both the grass factor and the soil-temperature proxy
       TSOI = 0.72*TAIR + 82.28 (no-soil path).  The port capped only the grass branch, so crop soil NO
       was too high whenever T2 > 30 C.  Fix: `soilnox_megan321` applies the cap as soilnox.F does.
FIX-8  ERA5-Land radiation de-accumulation at 00 and 01 UTC (MODERATE; found by the notebook test, Oct 2026).
       ERA5-Land accumulations run from 00 UTC, and the field valid at 00 UTC holds the 24 h ending at
       00 UTC (ECMWF, ERA5-Land data documentation).  gee/era5.hourly_flux did not subtract anything at
       00 UTC (so PAR at 00 UTC was the previous day's 24-h total / 3600 x 0.45, ~2700 W m-2 in summer) and
       subtracted the 00 UTC field at 01 UTC (PAR clipped to 0).  In SC 00 UTC is night (about 21 h local
       solar time), so leaf light stayed 0, but the canopy stability/lapse term saw the spurious radiation
       (+4 to +8 % light-independent ER in that sample) and the PAR maps/driver series were inflated.
       Fix: `hourly_flux_fixed` subtracts the previous hour always except at 01 UTC.
NOTE   Southern-Hemisphere growing season (deliberate, documented deviation, kept).
       growseason.f compares the day of year (1-366) with MMDD constants (1101, 0101, 0531) in the
       lat < -23 branch, so south of 23 S the Fortran growing season becomes DOY 101-366.  The port's
       soil.growseason(corrected=True) applies the intended 1 Nov - 31 May season.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage

from .engine import canopy, megvea, soil, speciation as spc, gamma as _gamma, species as _sp

MET_KEYS = ("temp", "par", "wind", "pres", "qv", "soilm")
# physically admissible ranges for the ERA5-Land-derived MEGAN fields (SI units used by the engine)
VALID_RANGE = {
    "temp": (150.0, 340.0),       # K (2 m air temperature)
    "par": (0.0, 1000.0),         # W m-2 (0.45 x shortwave down)
    "wind": (0.0, 75.0),          # m s-1
    "pres": (4.0e4, 1.1e5),       # Pa
    "qv": (0.0, 0.06),            # kg kg-1
    "soilm": (0.0, 1.0),          # m3 m-3
}


# --------------------------------------------------------------------------- FIX-1
def invalid_met_mask(met: dict) -> np.ndarray:
    """True where ANY engine met band is non-finite or outside its physical range."""
    bad = None
    for k in MET_KEYS:
        if k not in met or met[k] is None:
            continue
        x = np.asarray(met[k], dtype=np.float64)
        lo, hi = VALID_RANGE[k]
        b = ~np.isfinite(x) | (x < lo) | (x > hi)
        bad = b if bad is None else (bad | b)
    if bad is None:
        raise ValueError("met dict has none of the expected keys " + str(MET_KEYS))
    return bad


def nearest_valid_index(bad: np.ndarray):
    """(ii, jj) index arrays pointing every cell to its nearest valid cell (itself if valid)."""
    if bad.all():
        raise RuntimeError("FIX-1: no valid meteorology anywhere on the grid")
    _, (ii, jj) = ndimage.distance_transform_edt(bad, return_indices=True)
    return ii, jj


def fill_met_nearest(met: dict, bad: np.ndarray | None = None, keys=MET_KEYS) -> tuple[dict, np.ndarray]:
    """Return (filled_met, bad_mask).  All bands of an invalid cell come from ONE donor cell."""
    bad = invalid_met_mask(met) if bad is None else bad
    out = dict(met)
    if not bad.any():
        return out, bad
    ii, jj = nearest_valid_index(bad)
    for k in keys:
        if k not in met or met[k] is None:
            continue
        x = np.array(met[k], dtype=np.float64, copy=True)
        x[bad] = x[ii[bad], jj[bad]]
        out[k] = x
    return out, bad


def fill_field_nearest(field, lo=-np.inf, hi=np.inf) -> np.ndarray:
    """Nearest-valid fill for a single 2-D field (e.g. daily D_TEMP)."""
    x = np.array(field, dtype=np.float64, copy=True)
    bad = ~np.isfinite(x) | (x < lo) | (x > hi)
    if bad.any():
        ii, jj = nearest_valid_index(bad)
        x[bad] = x[ii[bad], jj[bad]]
    return x


def assert_valid_on_vegetation(met: dict, lai) -> None:
    """Raise if any cell with LAI > 0 still has invalid meteorology (guards against FIX-1 regressions)."""
    lai = np.nan_to_num(np.asarray(lai, dtype=np.float64), nan=0.0)
    bad = invalid_met_mask(met) & (lai > 0)
    if bad.any():
        raise RuntimeError(f"FIX-1: {int(bad.sum())} vegetated cells still have invalid meteorology")


# --------------------------------------------------------------------------- FIX-4
def fill_lai_gaps(lai_m, lai_prev, lai_next, veg_frac, veg_min=0.05) -> tuple[np.ndarray, np.ndarray]:
    """Fill vegetated cells whose monthly LAI is missing/zero from the adjacent valid months.

    Uses the mean of the previous and next month where both are valid, otherwise whichever is
    valid.  Cells without vegetation (veg_frac <= veg_min) are left untouched.  Returns
    (filled_lai, filled_mask)."""
    cur = np.nan_to_num(np.asarray(lai_m, dtype=np.float64), nan=0.0)
    p = np.nan_to_num(np.asarray(lai_prev, dtype=np.float64), nan=0.0)
    n = np.nan_to_num(np.asarray(lai_next, dtype=np.float64), nan=0.0)
    gap = (cur <= 0) & (np.asarray(veg_frac) > veg_min) & ((p > 0) | (n > 0))
    both = (p > 0) & (n > 0)
    repl = np.where(both, 0.5 * (p + n), np.where(p > 0, p, n))
    out = np.where(gap, repl, cur)
    return out, gap


# --------------------------------------------------------------------------- FIX-2 helpers
PAR_WM2_TO_PPFD = 4.5      # daymet.f L320 / megcan.f


def daily_means_from_samples(temp_samples, par_samples):
    """D_TEMP (K) and D_PPFD (umol m-2 s-1) from a stack of same-day samples (time on axis 0).

    Prefer a 24-hourly stack for D_TEMP (MEGAN daymet uses 24 records); with 3-hourly samples the
    mean of the 8 values is an unbiased estimator of the daily mean for a smooth diurnal cycle.
    D_PPFD only acts as a night gate in MEGANv3.21 (C1 is constant in GAMP)."""
    t = np.asarray(temp_samples, dtype=np.float64)
    p = np.asarray(par_samples, dtype=np.float64)
    return t.mean(axis=0), p.mean(axis=0) * PAR_WM2_TO_PPFD


# --------------------------------------------------------------------------- corrected timestep
def run_timestep_fixed(domain, ef, met, *, day, hour_utc, canopy_type_fractions,
                       d_temp, d_ppfd, laip, nlai=12, wilt=0.12, date_yyyyddd=None,
                       soiltype=None, strict=True):
    """Drop-in replacement for run.run_timestep with FIX-1, FIX-2 and FIX-3 applied.

    `met` holds temp, par, wind, pres, qv, soilm, lai (current-month LAI, already gap-filled).
    `d_temp`, `d_ppfd` are the daily means of the simulation day (required).
    `laip` is the previous month's LAI (required); `nlai` = 12 for monthly LAI.
    Returns the same dict as run.run_timestep plus 'filled' (cells whose met was replaced).
    """
    if d_temp is None or d_ppfd is None:
        raise ValueError("FIX-2: d_temp and d_ppfd (daily means of the simulation day) are required")
    if laip is None:
        raise ValueError("FIX-3: laip (previous-period LAI) is required")
    lat, lon = domain["LAT"], domain["LON"]
    lai = np.nan_to_num(np.asarray(met["lai"], dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    laip = np.nan_to_num(np.asarray(laip, dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)

    met_f, bad = fill_met_nearest(met)                                    # FIX-1
    if strict:
        assert_valid_on_vegetation(met_f, lai)
    d_temp = fill_field_nearest(d_temp, *VALID_RANGE["temp"])             # same rule for daily T
    d_ppfd = np.nan_to_num(np.asarray(d_ppfd, dtype=np.float64), nan=0.0)

    can = canopy.megcan(met_f["temp"], met_f["par"], met_f["wind"], met_f["pres"],
                        met_f["qv"], lat, lon, lai, canopy_type_fractions, day, hour_utc)
    gamsm = soil.gamma_sm(met_f["soilm"], wilt)
    er = megvea.emission_activity(can, laic=lai, laip=laip, d_temp=d_temp, d_ppfd=d_ppfd,
                                  gamsm=gamsm, nlai=nlai)                 # FIX-2, FIX-3

    gamno = None
    if date_yyyyddd is not None:
        kw = {}
        if soiltype is not None and "soilt" in met_f and met_f.get("soilt") is not None:
            kw = dict(soilt=met_f["soilt"], soilm=met_f["soilm"], soiltype=soiltype)
        cfno, cfnog = soil.soilnox(met_f["temp"], lai, lat, date_yyyyddd, **kw)
        gday, _ = soil.growseason(date_yyyyddd, lat)
        gamno = soil.gamma_no(cfno, cfnog, canopy_type_fractions, gday)

    compounds = spc.speciate_to_compounds(er, ef, gamno=gamno)
    classes = spc.compounds_to_classes(compounds)
    return {"compounds": compounds, "classes": classes, "er": er, "gamno": gamno,
            "filled": bad & (lai > 0)}


# --------------------------------------------------------------------------- FIX-5
EVT_NODATA = 65535


def fill_ecotype_nearest(src_nc: str, dst_nc: str, nodata: int = EVT_NODATA, var: str = "EVT") -> int:
    """Copy the ecotype NetCDF replacing no-data pixels by the nearest valid ecotype.

    Returns the number of pixels filled.  Ocean pixels are filled too, which is harmless because
    ocean cells carry no MapBiomas vegetation (zero growth-form fractions -> zero EF weight)."""
    from netCDF4 import Dataset
    import shutil
    shutil.copyfile(src_nc, dst_nc)
    with Dataset(dst_nc, "r+") as nc:
        evt = np.asarray(nc.variables[var][:])
        bad = evt == nodata
        if bad.any():
            ii, jj = nearest_valid_index(bad)
            evt = evt.copy(); evt[bad] = evt[ii[bad], jj[bad]]
            nc.variables[var][:] = evt
        return int(bad.sum())


# --------------------------------------------------------------------------- FIX-6
def gamtp_megan321(lai, sun_t, sha_t, sun_p, sha_p, sun_f, d_temp, d_ppfd, ldf, s, layers=5):
    """GAMTP exactly as megvea.f L725-736 (CDEA multiplies the sunlit light-dependent term only)."""
    g = _gamma
    _, vpgwt = _sp.canopy_nodes_weights(layers)
    cdea = g.gamma_cd(lai, layers)
    dt = np.asarray(d_temp, dtype=np.float64)[..., None]
    dp = np.asarray(d_ppfd, dtype=np.float64)[..., None]
    ldf_b = np.asarray(ldf, dtype=np.float64)[..., None]
    ea1 = (cdea * g.gamtld(sun_t, dt, s) * g.gamp(sun_p, dp) * sun_f
           + g.gamtld(sha_t, dt, s) * g.gamp(sha_p, dp) * (1.0 - sun_f))
    ea2 = g.gamtli(sun_t, s) * sun_f + g.gamtli(sha_t, s) * (1.0 - sun_f)
    return ((ea1 * ldf_b + ea2 * (1.0 - ldf_b)) * vpgwt).sum(axis=-1)


# --------------------------------------------------------------------------- FIX-7
def soilnox_megan321(tair, lai, lat, date_yyyyddd, *, soilt=None, soilm=None, soiltype=None,
                     precadj=1.0, corrected=True):
    """soil.soilnox with the TAIR <= 303 K cap applied before both branches, as in soilnox.F."""
    t = np.minimum(np.asarray(tair, dtype=np.float64), 315.0)
    t = np.minimum(t, 303.0)
    return _ORIG["soilnox"](t, lai, lat, date_yyyyddd, soilt=soilt, soilm=soilm, soiltype=soiltype,
                            precadj=precadj, corrected=corrected)


# --------------------------------------------------------------------------- FIX-8
def deaccumulate(acc_cur, acc_prev, hour_utc):
    """Hourly mean flux (W m-2) over (h-1, h] from ERA5-Land accumulations (J m-2) - numpy reference.

    acc_cur = field valid at hour h, acc_prev = field valid at h-1.  At 01 UTC the field already is the
    1-h accumulation; at 00 UTC it is the 24-h total ending at 00 UTC, so the previous hour (23 UTC of the
    previous day) must be subtracted, like at every other hour."""
    prev = 0.0 if int(hour_utc) == 1 else np.asarray(acc_prev, dtype=np.float64)
    return np.maximum(np.asarray(acc_cur, dtype=np.float64) - prev, 0.0) / 3600.0


def hourly_flux_fixed(ee, region, date):
    """Earth Engine version of `deaccumulate` for surface_solar_radiation_downwards (replaces era5.hourly_flux)."""
    from .gee import era5 as _era5
    d = ee.Date(date)
    col = ee.ImageCollection(_era5.COLLECTION).filterBounds(region)
    band = "surface_solar_radiation_downwards"
    cur = col.filterDate(d, d.advance(1, "hour")).first().select(band)
    prev = ee.Image(col.filterDate(d.advance(-1, "hour"), d).first()).select(band)
    prev_acc = ee.Image(ee.Algorithms.If(ee.Number(d.get("hour")).eq(1), ee.Image(0), prev))
    return cur.subtract(prev_acc).max(0).divide(3600.0).rename("SW_WM2")


_ORIG = {}
def apply_engine_patches():
    """Install FIX-6, FIX-7 and FIX-8 in the running engine (idempotent).  Returns the list of patches."""
    from .gee import era5 as _era5
    if "gamtp" not in _ORIG:
        _ORIG["gamtp"] = _gamma.gamtp
        _ORIG["soilnox"] = soil.soilnox
        _ORIG["hourly_flux"] = _era5.hourly_flux
    _gamma.gamtp = gamtp_megan321          # megvea.emission_activity calls gamma.gamtp at run time
    soil.soilnox = soilnox_megan321        # run_timestep_fixed calls soil.soilnox at run time
    _era5.hourly_flux = hourly_flux_fixed  # era5.megan_met_image calls hourly_flux at run time
    return ["FIX-6 gamtp_megan321", "FIX-7 soilnox_megan321", "FIX-8 hourly_flux_fixed"]


def patches_active() -> bool:
    from .gee import era5 as _era5
    return (_gamma.gamtp is gamtp_megan321 and soil.soilnox is soilnox_megan321
            and _era5.hourly_flux is hourly_flux_fixed)
'''
with open(os.path.join(ENGINE_SRC, 'megan_sc', 'fixes.py'), 'w', encoding='utf-8') as f: f.write(FIXES_PY)
FIXES_SHA = hashlib.sha256(FIXES_PY.encode('utf-8')).hexdigest()[:16]
ENV = dict(MGN=MGN, OMP_LIB=OMP_LIB, ENGINE_SRC=ENGINE_SRC, PYTHONDONTWRITEBYTECODE='1', RAW=RAW)
print('scripts in', PY, '| fixes.py sha256[:16] =', FIXES_SHA)

In [ ]:
# 5 · Fortran, SE_US: DAYMET -> MEGCAN -> MEGVEA (GAMSM/GAMBD off), MEGSEA (YL soil NO), mgn2mech; then MEGVEA with GAMSM/GAMBD on
FD = os.path.join(WORK, 'se_us', 'fort'); FD2 = os.path.join(WORK, 'se_us', 'fort_smbd')
if not os.path.exists(os.path.join(FD, 'MGNERS.ncf')):
    sh(f'{PY}/run_fortran_se_us.sh {FD} N N 46', env=ENV)
if not os.path.exists(os.path.join(FD, 'MGN201_tonhr.ncf')):
    sh(f'{PY}/run_fortran_se_us_soil_spec.sh {FD}', env=ENV)
if not os.path.exists(os.path.join(FD2, 'MGNERS.ncf')):
    sh(f'{PY}/run_fortran_se_us.sh {FD2} Y Y 46', env=dict(ENV, SMFILE=f'{FD}/MGNSEA_YL_soil.ncf'))
for f in sorted(glob.glob(f'{FD}/log_*.txt') + glob.glob(f'{FD2}/log_*.txt')):
    print(os.path.relpath(f, WORK), 'Normal Completion' in open(f, errors='ignore').read())

In [ ]:
# 6 · S0 harness check: our Fortran run (GAMSM/GAMBD off, DAYMET over 23 records, as in the official step3/step6
# scripts) vs the intermediate output shipped in Output/INT. Expected: bit-identical (max_abs = 0) for DAYMET, CANMET and
# all 19 MGNERS classes (benchmark/results/S0_harness_vs_shipped_reference.csv).
sys.path.insert(0, PY)
import numpy as np, pandas as pd
from ioapi import rd, compare
REF = os.path.join(MGN, 'Output', 'INT'); rows = []
pairs = [('DAYMET.ncf', 'DAYMET.SE_US.2018122.ncf', ['D_TEMP', 'D_PPFD', 'MaxT', 'MaxWS']),
         ('CANMET.ncf', 'CANMET.SE_US.2018122.ncf', ['SunleafTK', 'ShadeleafTK', 'SunPPFD', 'ShadePPFD', 'SunFrac']),
         ('MGNERS.ncf', 'MGNERS.SE_US.2018122.ncf', ['ISOP', 'MBO', 'MT_PINE', 'MT_ACYC', 'MT_CAMP', 'MT_SABI', 'MT_AROM',
          'NO', 'SQT_HR', 'SQT_LR', 'MEOH', 'ACTO', 'ETOH', 'ACID', 'LVOC', 'OXPROD', 'STRESS', 'OTHER', 'CO'])]
for ours, ref, vars_ in pairs:
    if not os.path.exists(os.path.join(REF, ref)):
        print('reference not in tarball:', ref); continue
    for v in vars_:
        a = rd(os.path.join(FD, ours), v); b = rd(os.path.join(REF, ref), v)
        n = min(a.shape[0], b.shape[0])
        rows.append(dict(file=ref, var=v, steps=n, **compare(a[:n], b[:n])))
S0 = pd.DataFrame(rows); S0.to_csv(os.path.join(RESULTS_DIR, 'S0_harness_vs_shipped_reference.csv'), index=False)
print(S0.to_string(float_format=lambda v: f'{v:.3g}'))
print('S0 bit-identical:', bool((S0.max_abs == 0).all()), f'({len(S0)} variables)')

In [ ]:
# 7 · Python vs Fortran, SE_US (each run 1-3 min) ----------------------------------------------------------
SE = os.path.join(WORK, 'se_us')
runs = [('se_us_bench_N_N.csv', FD, 'N', 'N', '0', {}),
        ('se_us_bench_patched_N_N.csv', FD, 'N', 'N', '1', {}),
        ('se_us_bench_patched_Y_Y.csv', FD2, 'Y', 'Y', '1', {'SMFILE': f'{FD}/MGNSEA_YL_soil.ncf'})]
for out, fd, sm, bd, patch, extra in runs:
    if not os.path.exists(os.path.join(RESULTS_DIR, out)):
        r = sh(f'python3 {PY}/bench_se_us.py {ENGINE_SRC} {MGN} {fd} {RESULTS_DIR}/{out} {sm} {bd} | tail -6', env=dict(ENV, MEGAN_PATCH=patch, **extra))
for out, patch in (('se_us_bench_soil_unpatched.csv', '0'), ('se_us_bench_soil_patched.csv', '1')):
    sh(f'python3 {PY}/bench_soil.py {RESULTS_DIR}/{out}', env=dict(ENV, FD=FD, MEGAN_PATCH=patch))
sh(f'python3 {PY}/bench_spec.py {RESULTS_DIR}/se_us_bench_speciation.csv | tail -4', env=dict(ENV, FD=FD))

In [ ]:
# 8 · Santa Catarina case: 15 Jan and 15 Jul 2019 (inputs ~170 MB per day; deleted after use) -------------
for mon, day in ((1, 15), (7, 15)):
    out = os.path.join(RESULTS_DIR, f'sc_bench_sc_m{mon:02d}.csv')
    if os.path.exists(out): continue
    D = os.path.join(WORK, 'sc', f'm{mon:02d}')
    sh(f'python3 {PY}/make_sc_inputs.py {ENGINE_SRC} {RAW} {MGN} {D} {mon} {day}', env=ENV)
    sh(f'{PY}/run_fortran_sc.sh {D}', env=ENV)
    sh(f'python3 {PY}/bench_sc.py {ENGINE_SRC} {MEGAN_DIR} {D} {out} | tail -30', env=ENV)
    shutil.rmtree(D, ignore_errors=True)

In [ ]:
# 9 · Summary and verdict ---------------------------------------------------------------------------------
# Criteria for the patched engine (float32 Fortran output, float64 Python): 99th percentile of the cell-hour relative
# difference <= 1e-4 (<= 5e-4 for the canopy temperatures/PPFD, whose iterative energy balance converges to ~1e-4) and
# state/domain sums within 1e-5. The original port is expected to FAIL S3/S4 (FIX-6) and the no-soil NO path (FIX-7).
def summ(f):
    d = pd.read_csv(os.path.join(RESULTS_DIR, f)); key = 'stage' if 'stage' in d else 'test'
    g = d.groupby(key).agg(max_rel=('max_rel', 'max'), p99_rel=('p99_rel', 'max'),
                           sum_dev=('sum_ratio', lambda x: float(np.nanmax(np.abs(x - 1))) if x.notna().any() else np.nan))
    if 'day_tonnes_ratio' in d:
        g['day_tonnes_dev'] = d.groupby(key)['day_tonnes_ratio'].apply(lambda x: float(np.nanmax(np.abs(x - 1))) if x.notna().any() else np.nan)
    g.insert(0, 'file', f); return g.reset_index().rename(columns={key: 'stage'})
files = ['se_us_bench_N_N.csv', 'se_us_bench_patched_N_N.csv', 'se_us_bench_patched_Y_Y.csv',
         'se_us_bench_soil_unpatched.csv', 'se_us_bench_soil_patched.csv', 'sc_bench_sc_m01.csv', 'sc_bench_sc_m07.csv']
SUM = pd.concat([summ(f) for f in files if os.path.exists(os.path.join(RESULTS_DIR, f))], ignore_index=True)
def verdict(r):
    s, f = r.stage, r.file
    if 'SH season corrected' in s:
        return 'deviation by design (SH season)'
    if f.startswith('sc_') and 'as Fortran' in s:
        return 'PASS' if r.p99_rel <= 1e-4 else 'see note (SH crop cells)'
    original = ('original port' in s) or f in ('se_us_bench_N_N.csv', 'se_us_bench_soil_unpatched.csv')
    lim = 5e-4 if s.startswith('S2') else 1e-4
    tdev = r.get('day_tonnes_dev', np.nan)
    ok = r.p99_rel <= lim and (pd.isna(r.sum_dev) or r.sum_dev <= 1e-5) and (pd.isna(tdev) or tdev <= 1e-5)
    if ok: return 'PASS'
    if original and s.startswith(('S3', 'S4')): return 'expected FAIL (port bug, FIX-6)'
    if original and 'nosoil' in s: return 'expected FAIL (port bug, FIX-7)'
    return 'FAIL'
SUM['verdict'] = SUM.apply(verdict, axis=1)
sp = pd.read_csv(os.path.join(RESULTS_DIR, 'se_us_bench_speciation.csv'))
SUM.loc[len(SUM)] = dict(file='se_us_bench_speciation.csv', stage='mgn2mech 201 species (t/h)', max_rel=sp.max_rel_cellhour.max(), p99_rel=np.nan,
                         sum_dev=float(np.nanmax(np.abs(sp.ratio - 1))), verdict='PASS' if np.nanmax(np.abs(sp.ratio - 1)) <= 1e-5 else 'FAIL')
SUM['fixes_sha256'] = FIXES_SHA
SUM.to_csv(os.path.join(RESULTS_DIR, 'benchmark_summary.csv'), index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 60)
print(SUM.to_string(index=False, float_format=lambda v: f'{v:.3g}'))
print('\nFAIL rows:', int((SUM.verdict == 'FAIL').sum()), '| results in', RESULTS_DIR)

**Reading the summary.** `PASS` rows are the evidence that the engine, as used in production, reproduces MEGANv3.21.
`expected FAIL` rows are the controls without the two engine patches (FIX-6: GAMTP operator precedence, light-dependent
fraction of every class; FIX-7: soil NO without soil data, air temperature not capped at 303 K). In the SC July case the
"as Fortran" soil-NO comparison differs only in crop cells, where MEGANv3.21's Southern-Hemisphere season gives a
negative fertiliser factor (negative GAMNO) that the port clips at 0; non-crop cells agree to ~1e-6. Production uses
the Nov-May season adjusted to the Southern Hemisphere, so this row is a documented deviation and not a port error.